In [1]:
from utils import *
from heuristique_glouton import *
from heuristique_itérative import *
from evaluation import *

In [2]:
df_ville,df_object,capacity=parse_ttp_file("a280_n279_bounded-strongly-corr_01.ttp")

In [3]:
df_ville

,X,Y
1,288,149
2,288,129
3,270,133
4,256,141
5,256,157
...,...,...
276,236,145
277,246,141
278,252,125
279,260,129


In [4]:
df_object

,Profit,Weight,City_Index
1,101,1,2
2,202,2,3
3,404,4,4
4,202,2,5
5,996,896,6
...,...,...,...
275,786,686,276
276,1572,1372,277
277,786,686,278
278,566,466,279


In [5]:
pi,obj_pris,poids_tot,dict_ville_objet_pris=algo_glouton(df_ville,df_object,capacity)

Loading:  96%|█████████▌| 24949/25936 [00:13<00:00, 1913.88weight unit/s, Current Weight=24949, Nb_ville=254]


In [6]:
pi2, obj_pris2,new_poids_tot, dict_ville_objet_pris2 = algo_iteratif(df_ville,df_object,capacity)

In [7]:
for ville in pi:
    print("Dans la ville ",ville," le voleur prend ",len(dict_ville_objet_pris[ville])," objet(s).")

Dans la ville  1  le voleur prend  0  objet(s).
Dans la ville  6  le voleur prend  0  objet(s).
Dans la ville  7  le voleur prend  0  objet(s).
Dans la ville  178  le voleur prend  0  objet(s).
Dans la ville  99  le voleur prend  0  objet(s).
Dans la ville  96  le voleur prend  0  objet(s).
Dans la ville  97  le voleur prend  0  objet(s).
Dans la ville  98  le voleur prend  0  objet(s).
Dans la ville  93  le voleur prend  0  objet(s).
Dans la ville  94  le voleur prend  0  objet(s).
Dans la ville  95  le voleur prend  0  objet(s).
Dans la ville  76  le voleur prend  0  objet(s).
Dans la ville  117  le voleur prend  0  objet(s).
Dans la ville  115  le voleur prend  0  objet(s).
Dans la ville  116  le voleur prend  0  objet(s).
Dans la ville  64  le voleur prend  0  objet(s).
Dans la ville  56  le voleur prend  0  objet(s).
Dans la ville  55  le voleur prend  0  objet(s).
Dans la ville  54  le voleur prend  0  objet(s).
Dans la ville  53  le voleur prend  0  objet(s).
Dans la ville  38  

In [8]:
eval_non_lin(pi,df_ville,df_object,dict_ville_objet_pris,obj_pris,capacity)

BENEFICE :  39249
COUT :  6840.9321285460655


32408.067871453935

In [9]:
eval_non_lin(pi2,df_ville,df_object,dict_ville_objet_pris2,obj_pris2,capacity)

BENEFICE :  150737
COUT :  34002.542994979805


116734.4570050202

In [10]:
eval_lin(pi,df_ville,df_object,dict_ville_objet_pris,obj_pris)

BENEFICE :  39249
COUT :  31281285.950658288


-31242036.950658288

In [11]:
eval_lin(pi2,df_ville,df_object,dict_ville_objet_pris2,obj_pris2)

BENEFICE :  150737
COUT :  0.0


150737.0

In [13]:
import gurobipy as gp

model=gp.Model()

# Create variables
list_index_ville=list(df_ville.index)
dict_obj_ville={}
list_obj=[]
for index_ville in list_index_ville:
    dict_obj_ville[index_ville]=[]
    list_index_obj=list(get_objects_of_ville(index_ville+1,df_object).index)
    for index_obj in list_index_obj:
        x=model.addVar(vtype="B",name="X_$index_obj$")
        dict_obj_ville[index_ville].append(x)
        list_obj.append(x)

matrix_distance={index_ville:calcul_distance_de_ville(index_ville,df_ville) for index_ville in list_index_ville}

pi=[]
for i in range(len(list_index_ville)):
    pi.append([])
    for j in range(len(list_index_ville)):
        pi[i].append(model.addVar(vtype="B",name="Pi_$i$-$j$"))

# Create constraints
model.addConstr(pi[0][0]==1)
for i in range(len(list_index_ville)):
    model.addConstr(sum([pi[i][j] for j in range(len(pi[i]))])==1)
    model.addConstr(sum([pi[j][i] for j in range(len(pi[i]))])==1)

list_poids=[]
for i in range(len(list_obj)):
    list_poids.append(df_object.iloc[i]["Weight"])

model.addConstr(sum([list_poids[i]*list_obj[i] for i in range(len(list_poids))]) <= capacity)

z=[]
for i in range(len(list_index_ville)-1):
    z.append([])
    for j in range(len(list_index_ville)):
        z[i].append([])
        for k in range(len(list_index_ville)):
            z[i][j].append(model.addVar(vtype="B",name="Z_$i_$j_$k"))
            model.addConstr(z[i][j][k]<=pi[i][j])
            model.addConstr(z[i][j][k]<=pi[i+1][k])
            model.addConstr(z[i][j][k]>=pi[i][j]+pi[i+1][k]-1)
            

list_benefits=[]
for i in range(len(list_obj)):
    list_benefits.append(df_object.iloc[i]["Profit"])

benefit_tot=sum(list_benefits[i]*list_obj[i] for i in range(len(list_benefits)))

total_distance_expr =0
poids_actu=0

for i in range(len(list_index_ville)-1):
    for j in range(len(list_index_ville)):
        for k in range(len(list_index_ville)):
            total_distance_expr+=z[i][j][k]*matrix_distance[j+1][k+1]*poids_actu
        list_index_obj=list(get_objects_of_ville(j+1,df_object).index)
        poids_actu+=sum([list_obj[obj_index-1]*list_poids[obj_index-1]*pi[i][j] for obj_index in list_index_obj])
            


# Optional: Set this as an objective or constraint
model.setObjective(benefit_tot-(1/1000)*total_distance_expr, gp.GRB.MAXIMIZE)  # Minimizing distance

KeyboardInterrupt: 

In [ ]:
import model_optimization
model = model_optimization.optimize_model(df_ville, df_object, capacity)

NameError: name 'df_ville' is not defined

In [ ]:
model.optimize()